# Kandinsky V2A — G4 / A100 GPU Comparison (Colab)

This notebook benchmarks Kandinsky V2A on one assigned Colab GPU. Run it once on G4 and once on A100 80 GB to compare runtime and memory before the AWS run. It reruns one representative five-second scene with Lite Distill and full Lite.

The earlier three-scene A100 pilot already exists. This run adds the missing peak GPU/system-memory samples and a fuller environment record; it does not retrain or change the model.

## Fixed reference
- Source: Wikimedia Commons “Big City Life”, marked CC0 1.0.
- Exact five-second window, prompt, seed, and Kandinsky source commit from the previous A100 report.
- Lite Distill and full Lite; SDPA; no prompt beautifier; 5 seconds.
- Run separately on G4 and A100 80 GB with all other settings unchanged; each ZIP records the actual GPU.
- The notebook exports the input window, generated outputs, configs, environment, resource samples, and hashes. Do not put model weights or secrets in the ZIP.

## Step 1 — Check the Colab runtime

Select G4 for the first run, then run this same notebook again on A100 80 GB for the paired comparison. H100 80 GB is also accepted and recorded if you want the later AWS p5.4xlarge reference. Each Colab session uses one GPU; keep every other setting unchanged. This cell identifies the actual GPU and records available disk/RAM before downloading anything.

In [ ]:
import os, sys, shutil, subprocess, json, time, hashlib, zipfile, urllib.request, csv, io, platform
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import Audio, display

print("Python:", sys.version)
if sys.version_info < (3, 13):
    raise RuntimeError("Select the Python 3.13+ Colab runtime used by the working Kandinsky notebook.")
if not shutil.which("nvidia-smi"):
    raise RuntimeError("No GPU runtime. Select G4, A100 80 GB, or H100 80 GB, then reconnect.")

gpu_probe = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,compute_cap",
     "--format=csv,noheader,nounits"],
    capture_output=True, text=True, check=False,
)
print(gpu_probe.stdout or gpu_probe.stderr)
if gpu_probe.returncode or not gpu_probe.stdout.strip():
    raise RuntimeError("GPU inspection failed. Reconnect to a GPU runtime.")
gpu_fields = [x.strip() for x in gpu_probe.stdout.strip().splitlines()[0].split(",")]
gpu_name = gpu_fields[0]
gpu_memory_mib = int(gpu_fields[1])
gpu_name_upper = gpu_name.upper()
if "H100" in gpu_name_upper and gpu_memory_mib >= 76000:
    gpu_profile = "h100_80gb"
elif "A100" in gpu_name_upper and gpu_memory_mib >= 76000:
    gpu_profile = "a100_80gb"
elif "RTX PRO 6000" in gpu_name_upper and gpu_memory_mib >= 76000:
    gpu_profile = "g4_rtx_pro_6000_blackwell"
else:
    raise RuntimeError(
        "Unsupported GPU for this comparison. Select G4 (RTX PRO 6000 Blackwell), "
        "A100 80 GB, or H100 80 GB. Detected: "
        f"{gpu_name}, {gpu_memory_mib} MiB"
    )

storage_candidates = [Path("/content/local-scratch"), Path("/content")]
CACHE_ROOT = max(
    (p for p in storage_candidates if p.exists()),
    key=lambda p: shutil.disk_usage(p).free,
)
WORK = CACHE_ROOT / "kandinsky_v2a_aws_baseline"
WORK.mkdir(parents=True, exist_ok=True)
free_gib = shutil.disk_usage(CACHE_ROOT).free / (1024**3)
print("GPU profile:", gpu_profile)
print("Detected GPU:", gpu_name, "|", gpu_memory_mib, "MiB")
print("Model/output storage:", CACHE_ROOT)
print(f"Free storage: {free_gib:.1f} GiB")
if free_gib < 45:
    raise RuntimeError("Free at least 45 GiB on the selected Colab storage before downloading both Lite checkpoints.")
ram_lines = Path("/proc/meminfo").read_text().splitlines()
ram_total_gib = int(next(x for x in ram_lines if x.startswith("MemTotal:")).split()[1]) / (1024**2)
print(f"System RAM: {ram_total_gib:.1f} GiB")

## Step 2 — Recreate the exact Kandinsky source environment

The previous report recorded source commit 7ceeefad7fb625011f7db602c43893a989f19c15. We check out that commit instead of silently following a newer upstream revision. Setup is performed once in this notebook runtime.

In [ ]:
SOURCE_COMMIT = "7ceeefad7fb625011f7db602c43893a989f19c15"
REPO = Path("/content/kandinsky-v2a-aws-baseline-src")

def run_checked(command, cwd=None, label="command"):
    result = subprocess.run(command, cwd=cwd, capture_output=True, text=True)
    if result.returncode:
        print(result.stdout[-2500:])
        print(result.stderr[-5000:])
        raise RuntimeError(f"{label} failed.")
    return result

if not (REPO / ".git").exists():
    clone = run_checked(
        ["git", "clone", "--filter=blob:none",
         "https://github.com/kandinskylab/kandinsky-6.git", str(REPO)],
        label="Kandinsky clone",
    )

run_checked(["git", "fetch", "origin", SOURCE_COMMIT], cwd=REPO, label="fetch pinned commit")
run_checked(["git", "checkout", "--detach", SOURCE_COMMIT], cwd=REPO, label="checkout pinned commit")
actual_commit = run_checked(["git", "rev-parse", "HEAD"], cwd=REPO, label="read source commit").stdout.strip()
if actual_commit != SOURCE_COMMIT:
    raise RuntimeError(f"Wrong source revision: {actual_commit}")
print("Pinned Kandinsky commit:", actual_commit)

run_checked([sys.executable, "-m", "pip", "-q", "install", "uv"], label="install uv")
if not Path("/usr/local/bin/just").exists():
    run_checked(
        ["bash", "-lc", "curl -fsSL https://just.systems/install.sh | bash -s -- --to /usr/local/bin"],
        label="install just",
    )
setup = subprocess.run(
    ["uv", "sync", "--extra", "consumer", "--inexact", "--no-dev"],
    cwd=REPO, capture_output=True, text=True,
)
if setup.returncode:
    print(setup.stdout[-2500:])
    print(setup.stderr[-5000:])
    raise RuntimeError("Kandinsky dependency setup failed.")
PY = REPO / ".venv" / "bin" / "python"
print("Environment ready:", PY)

## Step 3 — Rebuild and verify the exact CC0 input window

The previous pilot used the first five seconds of the Commons video. The source-file hash must match the earlier pilot. We also record the prepared-window hash. If a newer FFmpeg encodes the same crop to different bytes, the ZIP's window becomes the exact input for the later AWS run.

In [ ]:
SOURCE_PAGE = "https://commons.wikimedia.org/wiki/File:Big_City_Life.webm"
SOURCE_URL = "https://commons.wikimedia.org/wiki/Special:Redirect/file/Big_City_Life.webm"
EXPECTED_SOURCE_SHA256 = "2834fbc411cc59f06a63f9ac7ce9ba5b748b67242d1b36a4b18043a070a7966a"
EXPECTED_WINDOW_SHA256 = "9aa4cb5e674c7b9860f1d87ba59806f8211dcc3d4b418f7e0ab258ba4b5d49cc"
PROMPT = "Several pedestrians walking on a city pavement, with soft footsteps and distant urban movement. No music or speech."
SEED = 42
DURATION_SECONDS = 5
WINDOW_START_SECONDS = 0.0

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

INPUT_DIR = WORK / "input"
OUTPUT_DIR = WORK / "outputs"
INPUT_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
source_path = INPUT_DIR / "Big_City_Life.webm"
request = urllib.request.Request(
    SOURCE_URL,
    headers={"User-Agent": "Kandinsky-V2A-AWS-Baseline/1.0 (research reproducibility)"},
)
with urllib.request.urlopen(request, timeout=180) as response:
    source_path.write_bytes(response.read())
print("Source SHA-256:", sha256_file(source_path))
if sha256_file(source_path) != EXPECTED_SOURCE_SHA256:
    raise RuntimeError("The source-file hash differs from the previous pilot. Stop and inspect the download.")

window_path = INPUT_DIR / "city_walkers_00_05_silent.mp4"
prep = run_checked([
    "ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
    "-ss", str(WINDOW_START_SECONDS), "-i", str(source_path),
    "-t", str(DURATION_SECONDS), "-map", "0:v:0", "-an",
    "-vf", "fps=24,scale='min(960,iw)':-2",
    "-c:v", "libx264", "-preset", "veryfast", "-crf", "25",
    "-pix_fmt", "yuv420p", str(window_path),
], label="prepare exact video window")
window_sha256 = sha256_file(window_path)
print("Window SHA-256:", window_sha256)
if window_sha256 != EXPECTED_WINDOW_SHA256:
    print("Note: the prepared MP4 bytes differ from the previous report, possibly due to FFmpeg. This ZIP's window file and hash will be the exact AWS comparison input.")
probe = run_checked([
    "ffprobe", "-v", "error", "-show_entries", "format=duration",
    "-of", "default=noprint_wrappers=1:nokey=1", str(window_path),
], label="inspect video window")
print("Window duration:", float(probe.stdout.strip()), "seconds")
print("Prompt:", PROMPT)
print("Seed:", SEED)

## Step 4 — Download the two checkpoints and record the runtime

This does not train or fine-tune anything. Checkpoints stay in the Colab cache and are not included in the result bundle.

In [ ]:
KANDINSKY_HOME = CACHE_ROOT / "kandinsky-model-cache"
os.environ["KANDINSKY_HOME"] = str(KANDINSKY_HOME)
run_checked(["just", "download", "lite-distill", "lite"], cwd=REPO, label="download checkpoints")

# Record the immutable Hugging Face snapshot revisions used by this run.
WEIGHTS_DIR = KANDINSKY_HOME / "weights"
CHECKPOINT_REVISIONS = []
for ref_path in WEIGHTS_DIR.glob("models--*/refs/main"):
    revision = ref_path.read_text(encoding="utf-8").strip()
    cache_repo = ref_path.parent.parent
    snapshot_path = cache_repo / "snapshots" / revision
    CHECKPOINT_REVISIONS.append({
        "cache_repo": cache_repo.name,
        "revision": revision,
        "snapshot_exists": snapshot_path.exists(),
        "snapshot_path": str(snapshot_path),
    })
print("Cached checkpoint refs:", json.dumps(CHECKPOINT_REVISIONS, indent=2))

probe_code = (
    "import sys,json,torch; "
    "print(json.dumps({'python':sys.version,'torch':torch.__version__,"
    "'torch_cuda':torch.version.cuda,'cuda_available':torch.cuda.is_available(),"
    "'gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,"
    "'gpu_capability':torch.cuda.get_device_capability(0) if torch.cuda.is_available() else None}))"
)
torch_info = json.loads(run_checked([str(PY), "-c", probe_code], label="inspect PyTorch").stdout)
freeze = run_checked([str(PY), "-m", "pip", "freeze"], cwd=REPO, label="package freeze").stdout
(WORK / "environment_packages.txt").write_text(freeze, encoding="utf-8")
ffmpeg_version = subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True).stdout.splitlines()[0]
environment = {
    "captured_at_utc": datetime.now(timezone.utc).isoformat(),
    "python_notebook": sys.version,
    "platform": platform.platform(),
    "gpu_nvidia_smi": gpu_probe.stdout.strip(),
    "torch": torch_info,
    "ffmpeg_version": ffmpeg_version,
    "source_commit": SOURCE_COMMIT,
    "cache_path": str(KANDINSKY_HOME),
}
(WORK / "environment.json").write_text(json.dumps(environment, indent=2), encoding="utf-8")
print(json.dumps(environment, indent=2))
print("Recorded", len(freeze.splitlines()), "installed package entries.")

## Step 5 — Run the two GPU reference jobs and sample memory

Run this notebook once on G4 and once on A100 80 GB to compare them. The same video, prompt, seed, checkpoints, and SDPA settings are used in both sessions. H100 80 GB is also accepted as a separate AWS p5.4xlarge reference. The wall time includes CLI startup and model loading, matching the earlier end-to-end timing style. GPU and system memory are sampled about once per second during each generation.

In [ ]:
MODELS = [
    {"id": "lite_distill", "checkpoint": "lite-distill", "offload": "module"},
    {"id": "lite_full", "checkpoint": "lite", "offload": "none"},
]
POLL_SECONDS = 1.0
RESOURCE_DIR = WORK / "resources"
RESOURCE_DIR.mkdir(exist_ok=True)

def system_memory_mib():
    values = {}
    for line in Path("/proc/meminfo").read_text().splitlines():
        key, rest = line.split(":", 1)
        values[key] = int(rest.strip().split()[0])
    total = values["MemTotal"] / 1024
    available = values["MemAvailable"] / 1024
    return round(total, 1), round(total - available, 1)

def sample_resources(elapsed):
    query = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,memory.used,utilization.gpu",
         "--format=csv,noheader,nounits"],
        capture_output=True, text=True, check=False,
    )
    row = next(csv.reader([query.stdout.strip()])) if query.returncode == 0 and query.stdout.strip() else []
    ram_total, ram_used = system_memory_mib()
    return {
        "elapsed_seconds": round(elapsed, 2),
        "gpu_name": row[0].strip() if len(row) > 0 else "",
        "gpu_memory_total_mib": int(row[1]) if len(row) > 1 else None,
        "gpu_memory_used_mib": int(row[2]) if len(row) > 2 else None,
        "gpu_utilization_percent": int(row[3]) if len(row) > 3 else None,
        "system_memory_total_mib": ram_total,
        "system_memory_used_mib": ram_used,
    }

def run_generation(model):
    config_path = REPO / f"aws_baseline_{model['id']}.yaml"
    config_path.write_text(
        f"""checkpoint: {model['checkpoint']}
cache:
  mode: none
offload:
  strategy: {model['offload']}
compile:
  strategy: none
attention:
  engine: sdpa
text_embedder:
  quantized_qwen: false
beautifier:
  name: none
""",
        encoding="utf-8",
    )
    label = model["id"]
    output_mp4 = OUTPUT_DIR / f"city_walkers__{label}.mp4"
    output_wav = OUTPUT_DIR / f"city_walkers__{label}.wav"
    log_path = OUTPUT_DIR / f"{label}.log"
    command = [
        "just", "generate", PROMPT,
        "--config", str(config_path),
        "--attention-engine", "sdpa",
        "--video-file", str(window_path),
        "--time-length", str(DURATION_SECONDS),
        "--out", str(output_mp4),
        "--seed", str(SEED),
    ]
    samples = []
    started = time.monotonic()
    with open(log_path, "w", encoding="utf-8") as log:
        process = subprocess.Popen(command, cwd=REPO, stdout=log, stderr=subprocess.STDOUT, text=True)
        while process.poll() is None:
            samples.append(sample_resources(time.monotonic() - started))
            time.sleep(POLL_SECONDS)
        return_code = process.wait()
    elapsed = round(time.monotonic() - started, 2)
    samples.append(sample_resources(elapsed))
    sample_path = RESOURCE_DIR / f"{label}_samples.csv"
    with open(sample_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(samples[0].keys()))
        writer.writeheader()
        writer.writerows(samples)
    if return_code != 0 or not output_mp4.is_file():
        print(log_path.read_text(errors="replace")[-7000:])
        raise RuntimeError(f"{label} generation failed; inspect the log above.")
    audio_result = subprocess.run(
        ["ffmpeg", "-hide_banner", "-loglevel", "error", "-y", "-i", str(output_mp4),
         "-vn", "-ac", "2", "-ar", "44100", "-c:a", "pcm_s16le", str(output_wav)],
        capture_output=True, text=True,
    )
    if audio_result.returncode or not output_wav.is_file():
        raise RuntimeError("WAV extraction failed: " + audio_result.stderr[-2000:])
    return {
        "candidate_id": f"city_walkers__{label}",
        "checkpoint": model["checkpoint"],
        "config": config_path.name,
        "command": command,
        "wall_seconds_including_process_and_model_load": elapsed,
        "gpu_memory_peak_mib": max((x["gpu_memory_used_mib"] or 0) for x in samples),
        "system_memory_peak_mib": max(x["system_memory_used_mib"] for x in samples),
        "video_file": output_mp4.name,
        "audio_file": output_wav.name,
        "video_sha256": sha256_file(output_mp4),
        "audio_sha256": sha256_file(output_wav),
        "resource_samples_csv": sample_path.name,
        "log_file": log_path.name,
    }

CANDIDATES = []
for model in MODELS:
    print(f"Generating {model['id']}...")
    CANDIDATES.append(run_generation(model))
    print(CANDIDATES[-1]["wall_seconds_including_process_and_model_load"], "seconds | peak GPU",
          CANDIDATES[-1]["gpu_memory_peak_mib"], "MiB | peak system RAM",
          CANDIDATES[-1]["system_memory_peak_mib"], "MiB")
print("Both reference jobs completed.")

## Step 6 — Listen and record your impression

The previous report’s numerical ratings were blank. If you have time, enter the ratings after listening. Use the same 1–5 rubric as before; these are your observations, not an automated score.

In [ ]:
for item in CANDIDATES:
    print(item["candidate_id"])
    wav_path = OUTPUT_DIR / item["audio_file"]
    display(Audio(filename=str(wav_path)))

LISTENING = {
    item["candidate_id"]: {
        "audio_quality_1_to_5": None,
        "naturalness_1_to_5": None,
        "artifacts_or_noise_1_to_5": None,
        "scene_match_1_to_5": None,
        "sync_1_to_5": None,
        "keep": None,
        "notes": "",
    }
    for item in CANDIDATES
}
print("After listening, fill ratings in LISTENING and rerun the final bundle cell.")

## Step 7 — Export the GPU-specific comparison package

The ZIP filename includes the detected GPU profile. It contains the exact silent input window, both generated candidates, resource samples, full package list, logs, configs, hashes, and a JSON manifest. Run on G4 and A100 separately and keep both ZIPs for the comparison. Model weights and secrets are excluded.

In [ ]:
manifest = {
    "experiment": "kandinsky_v2a_g4_a100_colab_comparison_v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "purpose": "Colab GPU reference for later AWS V2A comparison; no training",
    "source_repository": "https://github.com/kandinskylab/kandinsky-6",
    "source_commit": SOURCE_COMMIT,
    "gpu_profile": gpu_profile,
    "gpu": gpu_probe.stdout.strip(),
    "comparison_note": "Run separately on G4 and A100 80 GB; compare the profile-specific manifests and resource samples.",
    "source": {
        "page": SOURCE_PAGE,
        "license_as_displayed_on_source_page": "CC0 1.0",
        "source_sha256": EXPECTED_SOURCE_SHA256,
        "window_start_seconds": WINDOW_START_SECONDS,
        "duration_seconds": DURATION_SECONDS,
        "window_file": window_path.name,
        "window_sha256": window_sha256,
        "previous_report_window_sha256": EXPECTED_WINDOW_SHA256,
    },
    "generation": {
        "prompt": PROMPT,
        "seed": SEED,
        "prompt_expansion": "disabled",
        "attention": "sdpa",
        "compile": "none",
        "sample_rate_hz": 44100,
    },
    "environment": environment,
    "checkpoint_cache_revisions": CHECKPOINT_REVISIONS,
    "candidates": CANDIDATES,
    "listening": LISTENING,
}
manifest_path = WORK / "manifest.json"
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")

bundle = WORK / f"kandinsky_v2a_gpu_comparison_{gpu_profile}.zip"
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as archive:
    archive.write(manifest_path, "manifest.json")
    archive.write(WORK / "environment_packages.txt", "environment_packages.txt")
    archive.write(window_path, "input/" + window_path.name)
    for item in CANDIDATES:
        config_path = REPO / item["config"]
        archive.write(config_path, "configs/" + config_path.name)
    for p in OUTPUT_DIR.iterdir():
        if p.is_file():
            archive.write(p, "outputs/" + p.name)
    for p in RESOURCE_DIR.glob("*.csv"):
        archive.write(p, "resources/" + p.name)
print("Ready:", bundle, "|", f"{bundle.stat().st_size / (1024**2):.2f} MiB")
from google.colab import files
files.download(str(bundle))

## What this gives us before AWS

This captures a GPU-specific reference: exact input bytes, pinned source revision, both checkpoint runs, peak GPU/system memory, environment packages, logs, wall time, and output hashes. Run on G4 and A100 80 GB in separate sessions, then compare the two ZIP manifests. For the later AWS p5.4xlarge run, use the same input window, prompt, seed, checkpoint, and config.

The measured wall time includes process startup and model loading. This is an end-to-end CLI benchmark, not warm inference latency for a persistent server. We will measure warm service latency separately only after the single-run AWS comparison.